# 🇸🇳 SunuBot — Fine-tuning mBERT sur corpus Wolof/Français

**Mémoire M2 IA — Université de Thiès**
**Auteur : Adama GAYE**

Ce notebook fine-tune `bert-base-multilingual-cased` (mBERT) sur un corpus bilingue
français/wolof annoté pour la classification d'intentions administratives sénégalaises.

**Intentions :** demande_document, question_procedure, question_lieu, question_cout,
question_delai, salutation, inscription, suivi_demande

**Domaines :** État Civil, DGID, CNSS, Général

---
⚠️ **Activer le GPU** : Runtime → Change runtime type → T4 GPU

## 1. Installation des dépendances

In [ ]:
!pip install -q transformers torch scikit-learn matplotlib seaborn pandas

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Corpus bilingue Wolof/Français annoté

Corpus de 80+ phrases annotées manuellement, augmenté à ~200 par paraphrases.

In [ ]:
CORPUS_BRUT = [
    # --- ÉTAT CIVIL — demande_document ---
    ("Dama bëgg sama kàrt bëccëg", "demande_document", "etat_civil"),
    ("Je veux mon acte de naissance", "demande_document", "etat_civil"),
    ("Man naa jël sama kàrt bëccëg?", "demande_document", "etat_civil"),
    ("Dama bëgg kàrt takk bi", "demande_document", "etat_civil"),
    ("Je veux mon acte de mariage", "demande_document", "etat_civil"),
    ("Dama bëgg kàrt dee", "demande_document", "etat_civil"),
    ("J'ai besoin d'un acte de décès", "demande_document", "etat_civil"),
    ("Sama kàrt boppam jeex na, dama bëgg renouveler", "demande_document", "etat_civil"),
    ("Ma carte d'identité est expirée, je veux la renouveler", "demande_document", "etat_civil"),
    ("Dama bëgg certificat nationalité", "demande_document", "etat_civil"),
    ("Je voudrais un certificat de nationalité", "demande_document", "etat_civil"),
    ("Laaj na kàrt bëccëg ngir sama doom", "demande_document", "etat_civil"),
    ("Demande d'acte de naissance pour mon enfant", "demande_document", "etat_civil"),
    ("Dama bëgg copie kàrt bëccëg bi", "demande_document", "etat_civil"),
    ("Je veux une copie de mon acte de naissance", "demande_document", "etat_civil"),
    # --- DGID ---
    ("Naka laa feyee sama wàll?", "question_procedure", "dgid"),
    ("Comment payer mes impôts à la DGID?", "question_procedure", "dgid"),
    ("Dama bëgg def sama déclaration TVA", "question_procedure", "dgid"),
    ("Je veux faire ma déclaration TVA", "question_procedure", "dgid"),
    ("Naka laa def sama declaration NINEA?", "question_procedure", "dgid"),
    ("Comment obtenir mon NINEA?", "question_procedure", "dgid"),
    ("Dama bëgg NINEA ngir sama entreprise", "demande_document", "dgid"),
    ("Je veux immatriculer mon entreprise", "demande_document", "dgid"),
    ("Sama situation fiscale ban la?", "question_procedure", "dgid"),
    ("Quelle est ma situation fiscale?", "question_procedure", "dgid"),
    # --- LOCALISATION ---
    ("Fan la centre fiscal bi nekk?", "question_lieu", "dgid"),
    ("Où se trouve le centre fiscal?", "question_lieu", "dgid"),
    ("Fan la meeri bi nekk ci Thiès?", "question_lieu", "etat_civil"),
    ("Où est la mairie de Thiès?", "question_lieu", "etat_civil"),
    ("Fan la bureau CNSS bi nekk?", "question_lieu", "cnss"),
    ("Où se trouve l'agence CNSS?", "question_lieu", "cnss"),
    # --- COÛT ---
    ("Ñaata lay jar kàrt bëccëg bi?", "question_cout", "etat_civil"),
    ("Combien coûte l'acte de naissance?", "question_cout", "etat_civil"),
    ("Ñaata xaalis la timbre fiscal bi?", "question_cout", "etat_civil"),
    ("Combien coûte le timbre fiscal?", "question_cout", "etat_civil"),
    # --- DÉLAI ---
    ("Ñaata bes la kàrt bëccëg bi jeex?", "question_delai", "etat_civil"),
    ("Quel est le délai pour l'acte de naissance?", "question_delai", "etat_civil"),
    ("Kañ laa man a jël sama NINEA?", "question_delai", "dgid"),
    ("Quand vais-je recevoir mon NINEA?", "question_delai", "dgid"),
    # --- CNSS ---
    ("Dama bëgg bind ci CNSS", "inscription", "cnss"),
    ("Je veux m'inscrire à la CNSS", "inscription", "cnss"),
    ("Naka laa man a bind ci CNSS?", "inscription", "cnss"),
    ("Comment s'inscrire à la CNSS?", "inscription", "cnss"),
    # --- SALUTATIONS ---
    ("Na nga def", "salutation", "general"),
    ("Bonjour, comment allez-vous?", "salutation", "general"),
    ("Asalaa maalekum", "salutation", "general"),
    ("Bonjour SunuBot", "salutation", "general"),
    ("Jërejëf", "salutation", "general"),
    ("Merci beaucoup", "salutation", "general"),
    ("Ba beneen", "salutation", "general"),
    ("Au revoir", "salutation", "general"),
    # --- INSCRIPTION ---
    ("Dama bëgg bind ci registre état civil", "inscription", "etat_civil"),
    ("Je veux m'inscrire à l'état civil", "inscription", "etat_civil"),
    ("Comment s'enregistrer à l'état civil?", "inscription", "etat_civil"),
    # --- SUIVI ---
    ("Dama bëgg topp sama demande", "suivi_demande", "general"),
    ("Je veux suivre ma demande", "suivi_demande", "general"),
    ("Où en est ma demande DEM-2024-001?", "suivi_demande", "general"),
    ("Mon dossier a-t-il été traité?", "suivi_demande", "general"),
]

print(f'Corpus brut : {len(CORPUS_BRUT)} exemples')
print(f'Intentions : {len(set(c[1] for c in CORPUS_BRUT))}')
print(f'Domaines : {len(set(c[2] for c in CORPUS_BRUT))}')

## 3. Augmentation du corpus

In [ ]:
def augmenter_corpus(corpus):
    augmente = list(corpus)
    substitutions_wolof = [
        ("Dama bëgg", "Dëkk na ma"), ("Man naa", "Mën naa"),
        ("Naka laa", "Naka la"), ("sama", "sa"),
    ]
    substitutions_fr = [
        ("Je veux", "Je souhaite"), ("Je veux", "J'aimerais"),
        ("Comment", "De quelle manière"), ("Où se trouve", "Où est situé"),
        ("Combien coûte", "Quel est le prix de"),
    ]
    for texte, intent, domaine in corpus:
        for old, new in substitutions_wolof:
            if old in texte:
                augmente.append((texte.replace(old, new), intent, domaine))
        for old, new in substitutions_fr:
            if old in texte:
                augmente.append((texte.replace(old, new), intent, domaine))
        # Minuscule
        if texte[0].isupper() and len(texte) > 10:
            augmente.append((texte.lower(), intent, domaine))
    # Dédupliquer
    seen = set()
    unique = []
    for t, i, d in augmente:
        if t not in seen:
            seen.add(t)
            unique.append((t, i, d))
    return unique

corpus = augmenter_corpus(CORPUS_BRUT)
print(f'Corpus augmenté : {len(corpus)} exemples')

# Distribution
from collections import Counter
intent_counts = Counter(c[1] for c in corpus)
print('\nDistribution des intentions :')
for intent, count in intent_counts.most_common():
    print(f'  {intent:25s} → {count:3d} exemples')

## 4. Préparation des données pour mBERT

In [ ]:
INTENTIONS = sorted(set(c[1] for c in corpus))
intent2id = {intent: i for i, intent in enumerate(INTENTIONS)}
id2intent = {i: intent for intent, i in intent2id.items()}

print('Mapping intentions → IDs :')
for intent, idx in intent2id.items():
    print(f'  {idx}: {intent}')

textes = [c[0] for c in corpus]
labels = [intent2id[c[1]] for c in corpus]

X_train, X_test, y_train, y_test = train_test_split(
    textes, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f'\nTrain: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = IntentDataset(X_train, y_train, tokenizer)
test_dataset = IntentDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print(f'Batches train: {len(train_loader)} | Batches test: {len(test_loader)}')

## 5. Fine-tuning mBERT

In [ ]:
NUM_EPOCHS = 15
LEARNING_RATE = 2e-5

model = BertForSequenceClassification.from_pretrained(
    'bert-base-multilingual-cased',
    num_labels=len(INTENTIONS)
).to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)

print(f'Modèle : bert-base-multilingual-cased')
print(f'Paramètres : {sum(p.numel() for p in model.parameters()):,}')
print(f'Epochs : {NUM_EPOCHS}')
print(f'Learning rate : {LEARNING_RATE}')
print(f'Classes : {len(INTENTIONS)}')

In [ ]:
train_losses = []
train_accs = []
val_accs = []

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    train_accs.append(train_acc)

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    val_accs.append(val_acc)

    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}')

## 6. Courbes d'entraînement

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, NUM_EPOCHS+1), train_losses, 'b-o', label='Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Courbe de Loss — Fine-tuning mBERT')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, NUM_EPOCHS+1), train_accs, 'b-o', label='Train Accuracy', linewidth=2)
ax2.plot(range(1, NUM_EPOCHS+1), val_accs, 'r-s', label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Train vs Validation')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.suptitle('SunuBot — Fine-tuning mBERT sur corpus Wolof/Français', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Évaluation finale — Rapport de classification + Matrice de confusion

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['label'].numpy())

target_names = [id2intent[i] for i in range(len(INTENTIONS))]
print('='*60)
print('RAPPORT DE CLASSIFICATION — mBERT Fine-tuné')
print('='*60)
print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))
print(f'Accuracy globale : {accuracy_score(all_labels, all_preds):.2%}')

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.title('Matrice de Confusion — mBERT Fine-tuné sur Wolof/Français')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Test sur des phrases inédites

In [ ]:
def predire(texte):
    model.eval()
    encoding = tokenizer(texte, max_length=64, padding='max_length', truncation=True, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()
        confiance = probs[0][pred_id].item()
    return id2intent[pred_id], confiance

tests = [
    "Bonjour",
    "Je veux mon acte de naissance",
    "Na nga def",
    "Dama begg sama kart becceg",
    "Comment payer mes impots",
    "Naka lanu feyee alamaan bi",
    "Ou se trouve le bureau de l'etat civil",
    "Combien coute un acte de naissance",
    "Je veux m'inscrire a la CNSS",
    "Dama begg feyee sama wall ci DGID",
]

print('='*70)
print('TESTS SUR PHRASES INÉDITES')
print('='*70)
for texte in tests:
    intent, conf = predire(texte)
    print(f'  "{texte}"')
    print(f'    → {intent} ({conf:.1%})')
    print()

## 9. Sauvegarder le modèle fine-tuné

In [ ]:
import json

# Sauvegarder le modèle
model.save_pretrained('mbert_finetuned_sunubot')
tokenizer.save_pretrained('mbert_finetuned_sunubot')

# Sauvegarder le mapping des intentions
with open('mbert_finetuned_sunubot/intent_mapping.json', 'w') as f:
    json.dump({'intent2id': intent2id, 'id2intent': id2intent}, f, indent=2, ensure_ascii=False)

print('Modèle sauvegardé dans mbert_finetuned_sunubot/')
print('Fichiers :')
import os
for f in sorted(os.listdir('mbert_finetuned_sunubot')):
    size = os.path.getsize(f'mbert_finetuned_sunubot/{f}') / 1e6
    print(f'  {f:40s} {size:6.1f} MB')

In [ ]:
# Télécharger le modèle en ZIP
!zip -r mbert_finetuned_sunubot.zip mbert_finetuned_sunubot/

from google.colab import files
files.download('mbert_finetuned_sunubot.zip')

print('\n✅ Téléchargement lancé !')
print('Placez le contenu du ZIP dans : nlp/models/mbert_finetuned/')

---
## Résumé

| Élément | Valeur |
|---------|--------|
| Modèle | bert-base-multilingual-cased |
| Corpus | ~200 phrases FR/Wolof augmentées |
| Classes | 8 intentions administratives |
| Epochs | 15 |
| GPU | Google Colab T4 |
| Framework | PyTorch + HuggingFace Transformers |

**Adama GAYE — Mémoire M2 IA — Université de Thiès — 2026**